## Adaptation and Predictions on New Data Using Pretrained Normative Models

##### *Authors: R. Cirstian, A. Bernas*

This notebook shows how to take an existing (pretrained) brain model
and apply it to new data.

You do NOT fully train a new model here.
Instead, you:
1. load a ready-made model
2. prepare your data in the right format
3. update the pre-trained model with your (training) data 
4. let the model compute deviation scores for your (test) subjects

Each cell below does one small, concrete step.

### Install the normative modelling toolkit

You only need to do this once per environment.  
We fix the version so that the behaviour is predictable
and matches the examples in this notebook.

In [1]:
# %pip install pcntoolkit==1.2.0.post1

### Check your Python version and location

This helps confirm that:
- the toolkit was installed in the same environment
- you are not accidentally running a different Python elsewhere

In [2]:
import sys
print(sys.version)
print(sys.executable)

3.12.13 | packaged by Anaconda, Inc. | (main, Mar 19 2026, 20:20:58) [GCC 14.3.0]
/home/preclineu/antber/.conda/envs/pcn130/bin/python


### Import the libraries we will use

In [3]:
import os
import pcntoolkit as ptk
import pandas as pd
import numpy as np
import zipfile
from importlib.metadata import version
print("pcntoolkit (metadata):", version("pcntoolkit"))

pcntoolkit (metadata): 1.2.0.post1


### Choose the pretrained model
Available models:
- BLRw_ct_DES_lifespan_67K_89sites.zip
- BLRw_fa_JHU_lifespan_24K_19sites.zip
- BLRw_fc_yeo17_lifespan_21K_40sites.zip
- BLRw_sa_DES_lifespan_37K_66sites.zip
- BLRw_sa_DK_lifespan_46K_59sites.zip
- BLRw_sc_lifespan_67K_89sites.zip
- HBR_Sb_ct_DES_lifespan_79K_100sites.zip
- HBR_Sb_ct_DK_lifespan_79K_100sites.zip
- HBR_Sb_sa_DES_lifespan_37K_66sites.zip
- HBR_Sb_sa_DK_lifespan_46K_59sites.zip
- HBR_Sb_sc_lifespan_79K_100sites.zip

Let's break down what these filenames mean:

- **BLRw**: Bayesian Linear Regression model (Fraza et al.)
- **HBR_Sb**: Hierarchical Bayesian Regression model using a SHASH likelihood (de Boer et al., 2024)
- **ct**: cortical thickness
- **fa**: fractional anisotropy
- **sc**: subcortical measures
- **DES**: Destrieux atlas parcellation
- **DK**: Desikan–Killiany atlas parcellation
- **67K / 79K / 24K**: number of subjects used to train the model
- **89sites / 100sites / 19sites**: number of sites in the training dataset

In [4]:
#Choose your model here. For this script, we will use the BLR of subcortical volumes from 67k subjects and 89 sites
model_name = 'BLRw_sc_lifespan_67K_89sites'

### Define where the data and model live on your system
We create the necessary directory structure for models and outputs. 
- top_dir: main project folder
- data_dir: where your input CSV files are
- model_dir: where the pretrained model is stored
  
The folders/directories are created automatically if they do not exist.

In [5]:
top_dir = os.path.expanduser('~/sfw/pu25_code')  # or use your preferred path
data_dir = os.path.join(top_dir,'data_test/')
root_dir = os.path.join(top_dir, 'test_pcntoolkit1.3.0')
model_dir = os.path.join(root_dir, 'models', model_name)
out_dir = model_dir + '_txfr'

# Create all directories
os.makedirs(model_dir, exist_ok=True)
os.makedirs(data_dir, exist_ok=True)
os.makedirs(out_dir, exist_ok=True)

print(f"Root directory: {root_dir}")
print(f"Data directory: {data_dir}")
print(f"Model directory: {model_dir}")
print(f"Output directory: {out_dir}")

Root directory: /home/preclineu/antber/sfw/pu25_code/test_pcntoolkit1.3.0
Data directory: /home/preclineu/antber/sfw/pu25_code/data_test/
Model directory: /home/preclineu/antber/sfw/pu25_code/test_pcntoolkit1.3.0/models/BLRw_sc_lifespan_67K_89sites
Output directory: /home/preclineu/antber/sfw/pu25_code/test_pcntoolkit1.3.0/models/BLRw_sc_lifespan_67K_89sites_txfr


## Download pretrained model

We now download the pretrained BLR model for subcortical volumes from the SURFdrive repository.


In [6]:
# Download model
model_zip = model_name + ".zip"
model_zip_path = os.path.join(model_dir, model_zip) #where the surfdrive zip file is downloaded

if not os.path.exists(model_zip_path):
    print("Downloading pre-trained model...")
    os.system(f'wget https://surfdrive.surf.nl/public.php/dav/files/Mb6mZyFmJeCaPcZ/zip/{model_zip} -O {model_zip_path}')
    print(f"✓ Downloaded to {model_zip_path}")
else:
    print(f"✓ Model file already exists")

# Extract the model (if not already done)
if not os.path.exists(os.path.join(model_dir, model_name)):
    print("Extracting model...")
    with zipfile.ZipFile(model_zip_path, 'r') as zip_ref:
        zip_ref.extractall(model_dir)
    print(f"✓ Extracted to {model_dir}")
else:
    print(f"✓ Model already extracted")

--2026-05-27 13:37:46--  https://surfdrive.surf.nl/public.php/dav/files/Mb6mZyFmJeCaPcZ/zip/BLRw_sc_lifespan_67K_89sites.zip
Resolving surfdrive.surf.nl (surfdrive.surf.nl)... 145.107.56.140, 145.107.8.140, 145.100.27.70, ...
Connecting to surfdrive.surf.nl (surfdrive.surf.nl)|145.107.56.140|:443... connected.
HTTP request sent, awaiting response... 

✓ Downloaded to /home/preclineu/antber/sfw/pu25_code/test_pcntoolkit1.3.0/models/BLRw_sc_lifespan_67K_89sites/BLRw_sc_lifespan_67K_89sites.zip
Extracting model...
✓ Extracted to /home/preclineu/antber/sfw/pu25_code/test_pcntoolkit1.3.0/models/BLRw_sc_lifespan_67K_89sites


200 OK
Length: 608714 (594K) [application/zip]
Saving to: ‘/home/preclineu/antber/sfw/pu25_code/test_pcntoolkit1.3.0/models/BLRw_sc_lifespan_67K_89sites/BLRw_sc_lifespan_67K_89sites.zip’

     0K .......... .......... .......... .......... ..........  8% 6.22M 0s
    50K .......... .......... .......... .......... .......... 16% 12.0M 0s
   100K .......... .......... .......... .......... .......... 25%  224M 0s
   150K .......... .......... .......... .......... .......... 33%  304M 0s
   200K .......... .......... .......... .......... .......... 42% 13.2M 0s
   250K .......... .......... .......... .......... .......... 50%  335M 0s
   300K .......... .......... .......... .......... .......... 58%  300M 0s
   350K .......... .......... .......... .......... .......... 67%  336M 0s
   400K .......... .......... .......... .......... .......... 75% 14.7M 0s
   450K .......... .......... .......... .......... .......... 84%  343M 0s
   500K .......... .......... .......... .......... 

### Load the pretrained Normative Model

This model already knows:
- which brain measures it was trained on
- which covariates it expects (e.g. age)
- which batch variables it uses (e.g. site, sex)  
  
We extract this information directly from the model
to avoid mismatches later.

In [7]:
model = ptk.NormativeModel.load(model_dir)
batch_effects = list(model.unique_batch_effects.keys()) #["site", "sex"]
covariates = model.covariates #["age"]
response_vars = model.response_vars

/home/preclineu/antber/.conda/envs/pcn130/lib/python3.12/site-packages/pcntoolkit/util/output.py:296: UserWarning: Process: 3649189 - 2026-05-27 13:37:46 - This model was saved with PCNtoolkit v1.1.2, but you are running v1.2.0.post1. Loading this model in v1.2.0.post1...
  warnings.warn(message, category)
/home/preclineu/antber/.conda/envs/pcn130/lib/python3.12/site-packages/pcntoolkit/util/output.py:296: UserWarning: Process: 3649189 - 2026-05-27 13:37:47 - This model was saved with PCNtoolkit v1.1.2, but you are running v1.2.0.post1. Loading this model in v1.2.0.post1...
  warnings.warn(message, category)


### Download the new data that we want to analyse  
  
- df_ad: a small set of subjects used to adapt the model
- df_te: the subjects we actually want results for  
  
Both files must contain the same columns and brain measures.

In [8]:
# Download and save data to data_dir
print("Downloading adaptation data from GitHub...")
df_ad = pd.read_csv('https://raw.githubusercontent.com/predictive-clinical-neuroscience/braincharts/master/docs/OpenNeuroTransfer_ct_ad.csv')
ad_path = os.path.join(data_dir, 'OpenNeuroTransfer_ct_ad.csv')
df_ad.to_csv(ad_path, index=False)
print(f"✓ Saved to: {ad_path} | Shape: {df_ad.shape}")

print("Downloading test data from GitHub...")
df_te = pd.read_csv('https://raw.githubusercontent.com/predictive-clinical-neuroscience/braincharts/master/docs/OpenNeuroTransfer_ct_te.csv')
te_path = os.path.join(data_dir, 'OpenNeuroTransfer_ct_te.csv')
df_te.to_csv(te_path, index=False)
print(f"✓ Saved to: {te_path} | Shape: {df_te.shape}")

✓ Saved to: /home/preclineu/antber/sfw/pu25_code/data_test/OpenNeuroTransfer_ct_ad.csv | Shape: (110, 250)
✓ Saved to: /home/preclineu/antber/sfw/pu25_code/data_test/OpenNeuroTransfer_ct_te.csv | Shape: (436, 250)


### Combine the two datasets into one table  
  
We do this because the toolkit expects a single dataframe.  
We will split it again later

In [9]:
# combine (we split again later)
df = pd.concat([df_ad, df_te], axis=0)

# adjust sex coding and add a subject id column
df['sex'] = df.apply(lambda x: {0: "F", 1: "M"}[x['sex']], axis=1)


### Keep only usable brain measures  
Not every brain measure in the model may be usable.   
Some measures may exist in the model structure.    
but were never successfully trained.    
  
Here we:
- keep only the measures that are fitted
- print which ones are skipped (for transparency)
- filter our data with only fitted measures
- print missing measures in our data (no worries, we can run the transfer with fewer responses)

In [10]:
# keep only response variables that are fitted in the pretrained model
fitted_response_vars = [
    rv for rv in response_vars
    if rv in model.response_vars and model[rv].is_fitted
]

skipped_response_vars = [
    rv for rv in response_vars
    if rv not in fitted_response_vars
]

print(f"Using {len(fitted_response_vars)} fitted response variables")
print("Skipped (not fitted):", skipped_response_vars)

# in df: Keep only fitted response vars + covariates + batch effects + subject ID
cols_to_keep = fitted_response_vars + covariates + batch_effects + ['sub_id']

# Filter to only columns that exist in df
cols_to_keep = [col for col in cols_to_keep if col in df.columns]

# Now filter the dataframe
df = df[cols_to_keep]

print(f"Filtered dataframe shape: {df.shape}")
print(f"Columns: {len(df.columns)}")
print(f"Missing from df: {[col for col in fitted_response_vars if col not in df.columns]}")


Using 32 fitted response variables
Skipped (not fitted): []
Filtered dataframe shape: (546, 35)
Columns: 35
Missing from df: ['log-WM-hypointensities']


In [11]:
print(len(cols_to_keep))
print(len(fitted_response_vars))
#print(fitted_response_vars)

35
32


### Filter out missing response variables

Some response variables in the dataset are completely missing (filled with zeros). 
That would mess up the regressions, we need to identify and remove these columns before analysis.

In [12]:
# Identify columns with only zeros (missing response variables)
zero_cols = df.columns[(df == 0).all()].tolist()

if zero_cols:
    print(f"Found {len(zero_cols)} missing response variables (all zeros):")
    for col in zero_cols:
        print(f"  - {col}")
    
    # Remove these columns
    df = df.drop(columns=zero_cols)
    print(f"\n✓ Filtered dataset shape: {df.shape}")
else:
    print("No missing response variables found")
    print(f"Dataset shape: {df.shape}")

#final  (= available in our df) response variables
available_response_vars = [col for col in df.columns if col not in covariates + batch_effects + ['sub_id']]
print(f"Available response variables: {len(available_response_vars)} out of {len(fitted_response_vars)}")

Found 2 missing response variables (all zeros):
  - Right-Thalamus-Proper
  - Left-Thalamus-Proper

✓ Filtered dataset shape: (546, 33)
Available response variables: 29 out of 32


### Create a NormData object
You can think of a NormData object as a container that holds your data in the format the normative model expects.

It does three important things:

1. Keeps all required information together. This includes:
   - the brain measures
   - the covariates (for example age)
   - the batch variables (for example site and sex)
   - the subject IDs
     
<!-- -->

2. Applies basic data cleaning automatically. During this step, the toolkit:
   - removes rows with missing values
   - removes extreme outliers
     
<!-- -->

3. Makes the data usable by the model.
   - the pretrained model does not work directly on a pandas dataframe
   - it expects the data to be wrapped in a NormData object instead

In [13]:
# configure norm data object
norm_data = ptk.NormData.from_dataframe(
    name="pu25_subcort_txfr", 
    dataframe=df,
    batch_effects=batch_effects,
    subject_ids='sub_id',
    response_vars = available_response_vars,
    covariates = covariates, 
    # optional arguments
    remove_Nan=True,
    remove_outliers=True,
    z_threshold=10  
)

Process: 3649189 - 2026-05-27 13:37:47 - Removed 0 NANs
Process: 3649189 - 2026-05-27 13:37:47 - Removed 1 outliers for Right-Inf-Lat-Vent
Process: 3649189 - 2026-05-27 13:37:47 - Removed 1 outliers for CSF
Process: 3649189 - 2026-05-27 13:37:47 - Removed 1 outliers for WM-hypointensities
Process: 3649189 - 2026-05-27 13:37:47 - Removed 3 outliers
Process: 3649189 - 2026-05-27 13:37:47 - Dataset "pu25_subcort_txfr" created.
    - 543 observations
    - 515 unique subjects
    - 1 covariates
    - 29 response variables
    - 2 batch effects:
    	site (6)
	sex (2)
    


### Split and apply the model  
  
- "train": used to adapt the model to this dataset
- "test": subjects we want final deviation scores for  
  
Here we use a simple 50/50 split for demonstration.

In [14]:
#  train test split
split = [0.5, 0.5]
train, test = norm_data.train_test_split(splits = split)

### Apply Transfer + Predict to the pretrained Normative Model  
  
When you apply the *transfer_predict* function below, it creates several result files for each subject and/or response variable:

*For both your training (transfered) and test (predicted) data*

- **Z-scores**  
  A number that shows how unusual a person’s brain measure is compared to the reference population,  
  after adjusting for things like age, sex, and site.

- **Centiles**  
  A ranking that shows where a person sits compared to others in the reference population.

- **Log-probabilities (logp)**  
  A measure of how likely the observed value is according to the model’s expectations.

- **Statistics**  
  Overall summary numbers that describe how well the model works as a whole,  
  not information about individual people.

In [15]:
# run transfer
model.transfer_predict(train, test, save_dir = out_dir)

Process: 3649189 - 2026-05-27 13:37:47 - Transferring models on 29 response variables.
Process: 3649189 - 2026-05-27 13:37:47 - Transferring model for CSF.
Process: 3649189 - 2026-05-27 13:37:47 - Transferring model for Right-VentralDC.
Process: 3649189 - 2026-05-27 13:37:47 - Transferring model for Left-Accumbens-area.
Process: 3649189 - 2026-05-27 13:37:47 - Transferring model for Right-Putamen.
Process: 3649189 - 2026-05-27 13:37:47 - Transferring model for Left-Cerebellum-Cortex.
Process: 3649189 - 2026-05-27 13:37:47 - Transferring model for Right-Accumbens-area.
Process: 3649189 - 2026-05-27 13:37:47 - Transferring model for Brain-Stem.
Process: 3649189 - 2026-05-27 13:37:47 - Transferring model for 3rd-Ventricle.
Process: 3649189 - 2026-05-27 13:37:47 - Transferring model for Right-Inf-Lat-Vent.
Process: 3649189 - 2026-05-27 13:37:47 - Transferring model for 4th-Ventricle.
Process: 3649189 - 2026-05-27 13:37:47 - Transferring model for Right-Amygdala.
Process: 3649189 - 2026-05-